In [1]:
!git clone https://github.com/phquyen/Reinforcement-Learning-Human-Feedback-RLHF-Implementation.git


fatal: destination path 'Reinforcement-Learning-Human-Feedback-RLHF-Implementation' already exists and is not an empty directory.


In [2]:
import os
os.chdir('/content/Reinforcement-Learning-Human-Feedback-RLHF-Implementation')

# Verify files are there
os.listdir()

['stf_base_model',
 'RLHF.ipynb',
 'alpaca_data.json',
 'trainer_output',
 '.git',
 'sft_base_model']

In [3]:
# library
from datasets import load_dataset, Dataset
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, GPT2TokenizerFast
from trl import SFTTrainer, SFTConfig
import torch
from sklearn.model_selection import train_test_split

In [4]:
# Load dataset
with open ("alpaca_data.json", "r") as file:
    data = json.load(file)

In [5]:
file_path = "alpaca_data.json"
df = pd.read_json(file_path)
df = df.head(2000)

In [6]:
# Split into train and test set
df_train, df_test = train_test_split(
    df,
    test_size = 0.1,
    random_state = 42
)

Data format:
1. Instruction: what we ask the model to do
2. Input: extra information to ask
3. Output: the expected answer the model should learn

In [7]:
# Prompt formatting
df_train["text"] = df_train.apply(
    lambda r: (
        f"### Instruction:\n{r['instruction']}\n"
        +
            (f"\n### Input:\n{r['input']}\n"
            if pd.notna(r["input"]) and str(r["input"]).strip() else ""
            )
        +
        f"\n### Response:\n{r['output']}"
),
axis = 1
)



In [8]:
# Tokenizer model base GPT2
model_name = "openai-community/gpt2"
tokenizer = GPT2TokenizerFast.from_pretrained(model_name) # loading specific model
tokenizer.pad_token = tokenizer.eos_token # making batch sequences the same length

# Modelling
model = AutoModelForCausalLM.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.pad_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# Train model with supervised fine-tunning
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to("cuda")

sft_seq_length = SFTConfig(
    max_length = 512
)

training_arg = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    fp16=True,
    # bf16=False,
    logging_steps=10,
    num_train_epochs=3,
    output_dir="./sft_output"
)

SFTtrainer = SFTTrainer(
    model = model,
    train_dataset = Dataset.from_pandas(df_train),
    processing_class = tokenizer,
    args = sft_seq_length
)

Adding EOS to train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

In [10]:
SFTtrainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.079403
20,2.628065
30,2.482760
40,2.360934
50,2.274128
60,2.426863
70,2.323816
80,2.292557
90,2.265874
100,2.274108


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=675, training_loss=2.125338995191786, metrics={'train_runtime': 485.0718, 'train_samples_per_second': 11.132, 'train_steps_per_second': 1.392, 'total_flos': 489012120576000.0, 'train_loss': 2.125338995191786})

In [11]:
SFTtrainer.save_model("sft_base_model")
tokenizer.save_pretrained("stf_base_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('stf_base_model/tokenizer_config.json', 'stf_base_model/tokenizer.json')

### Evaluate Model

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
baseline_model = AutoModelForCausalLM.from_pretrained("./sft_base_model").to(device)
baseline_tokenizer = AutoTokenizer.from_pretrained("./sft_base_model")

samples = df_test.sample(10, random_state=42).reset_index(drop=True)

results = []

for i, row in samples.iterrows():
    instruction = str(row["instruction"]).strip()
    input_text = str(row["input"]).strip() if pd.notna(row["input"]) else ""
    reference = str(row["output"]).strip()

    # Build prompt
    if input_text != "":
        prompt = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""
    else:
        prompt = f"""### Instruction:
{instruction}

### Response:
"""

    # Tokenize and move to device
    inputs = baseline_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate response
    output_ids = baseline_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=baseline_tokenizer.eos_token_id
    )

    generated_text = baseline_tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Extract only the response part
    if "### Response:" in generated_text:
        model_output = generated_text.split("### Response:")[-1].strip()
    else:
        model_output = generated_text.strip()

    # Save results
    results.append({
        "instruction": instruction,
        "input": input_text,
        "reference_output": reference,
        "model_output": model_output
    })

# Convert to dataframe
results_df = pd.DataFrame(results)

results_df

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

,instruction,input,reference_output,model_output
0,Give a reason why is reading difficult for a c...,,Reading can be difficult for some people due t...,The reason why reading difficult for a certain...
1,"Given a list of ingredients, come up with a dish","Salt, Pepper, Onion, Garlic",Sautéed Onion and Garlic with Salt and Pepper.,The salt and pepper are the main ingredients i...
2,"Find the synonym of the word ""loathe""",,"Hate, detest, despise, abhor, abominate, detest.",loathe
3,Provide a cause-and-effect explanation for the...,The pandemic has led to a significant increase...,The pandemic caused people to stay at home in ...,The pandemic has led to a significant increase...
4,Reword the sentence to use other words without...,She came late to the meeting.,She arrived tardily to the gathering.,She came late to the meeting.
5,Name all the elements in the periodic table wi...,,The elements in the periodic table with symbol...,"C = 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
6,Generate a story starter given this scenario.,"The dark, silent forest had been filled with r...","Lately, it seemed as if every corner of the da...",The story was told by a young girl named Kana....
7,Brainstorm ideas for a creative ensemble for a...,,"For a formal event, you could try a combinatio...",Ideas for a creative ensemble for a formal eve...
8,Come up with a riddle,,What's full of keys but can't open a single lo...,"The riddle is: ""What is the meaning of the wor..."
9,Compare and contrast water and oil.,,Water and oil are similar in that they are bot...,Water is a liquid that is heated by the sun an...
